In [ ]:
# Install python-docx for processing .docx files
!pip install python-docx

### Upload CV for Analysis

Use the button below to upload your CV (PDF or DOCX format recommended). Note: Direct support for older `.doc` files is complex in Colab; please convert them to `.pdf` or `.docx` for best results.

In [ ]:
# Install PyMuPDF for PDF processing
!pip install PyMuPDF

import spacy
import fitz
from google.colab import files
import os

# 1. Load the NLP Model
try:
    nlp = spacy.load("en_core_web_md")
except:
    !python -m spacy download en_core_web_md
    nlp = spacy.load("en_core_web_md")

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    return "".join([page.get_text() for page in doc])

def validate_is_job_ad(text):
    # Common words found in JDs but rarely at the top of a CV
    jd_markers = ["responsibilities", "qualifications", "requirements", "about the role", "company overview", "we are looking for"]
    text_lower = text.lower()
    return any(marker in text_lower for marker in jd_markers)

def strict_evaluate(jd_text, cv_text):
    jd_content = jd_text.lower()
    cv_content = cv_text.lower()

    skill_sets = {
        "AI_ENGINEER": ["python", "nlp", "tensorflow", "pytorch", "rag", "vector", "machine learning", "transformers"],
        "WEB_DEVELOPER": ["javascript", "react", "html", "css", "figma", "frontend", "ui", "ux", "tailwind"]
    }

    ai_hits = sum(1 for s in skill_sets["AI_ENGINEER"] if s in jd_content)
    web_hits = sum(1 for s in skill_sets["WEB_DEVELOPER"] if s in jd_content)

    job_type = "AI" if ai_hits >= web_hits else "WEB"
    relevant_skills = skill_sets["AI_ENGINEER"] if job_type == "AI" else skill_sets["WEB_DEVELOPER"]
    candidate_matches = [s for s in relevant_skills if s in cv_content]

    return job_type, len(candidate_matches), candidate_matches, (len(candidate_matches) >= 3)

# --- SHOWCASE EXECUTION LOOP ---
print("="*55)
print("   SKILLS GAP ANALYZER  ")
print("="*55)

# Loop for Job Advertisement Upload
while True:
    print("\nSTEP 1: Upload the JOB ADVERTISEMENT (PDF)")
    uploaded_jd = files.upload()
    jd_filename = list(uploaded_jd.keys())[0]
    jd_text = extract_text_from_pdf(jd_filename)

    if validate_is_job_ad(jd_text):
        print(f"✅ Success: {jd_filename} verified as a Job Advertisement.")
        break
    else:
        print("\n" + "!"*55)
        print(f"FILE TYPE ERROR: '{jd_filename}' does not appear to be a Job Ad.")
        print("Please try again and upload a valid Job Advertisement.")
        print("!"*55)
        # The loop continues, so files.upload() will pop up again automatically

# Step 2: Upload Candidate Resume (Simple upload, no loop needed here)
print("\nSTEP 2: Upload the CANDIDATE CV (PDF)")
uploaded_cv = files.upload()
cv_filename = list(uploaded_cv.keys())[0]
cv_text = extract_text_from_pdf(cv_filename)

# Step 3: Final Analysis
job_role, matches, skill_list, qualified = strict_evaluate(jd_text, cv_text)

print("\n" + "="*50)
print("\033[1;34m" + "            FINAL ANALYSIS REPORT          " + "\033[0m")
print("="*50)
print(f"DETECTED ROLE: {job_role}")
print(f"CANDIDATE:     {cv_filename}")
print(f"SKILLS FOUND:  {matches} ({', '.join(skill_list)})")
print("-" * 50)
print(f"VERDICT:       {'✅ QUALIFIED' if qualified else '❌ NOT QUALIFIED'}")
print("="*50 + "\n")

   SKILLS GAP ANALYZER  

STEP 1: Upload the JOB ADVERTISEMENT (PDF)


Saving Kiran_Kumar.pdf to Kiran_Kumar (2).pdf

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
FILE TYPE ERROR: 'Kiran_Kumar (2).pdf' does not appear to be a Job Ad.
Please try again and upload a valid Job Advertisement.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

STEP 1: Upload the JOB ADVERTISEMENT (PDF)


Saving Job_AI_Engineer.pdf to Job_AI_Engineer (5).pdf
✅ Success: Job_AI_Engineer (5).pdf verified as a Job Advertisement.

STEP 2: Upload the CANDIDATE CV (PDF)
